In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abdullahelnoory/bayyaaai-data/policies.txt
/kaggle/input/datasets/abdullahelnoory/bayyaaai-data/catalog.csv
/kaggle/input/datasets/abdullahelnoory/bayyaaai-data/faq.txt
/kaggle/input/datasets/abdullahelnoory/bayyaaai-data/faq.pdf
/kaggle/input/datasets/abdullahelnoory/bayyaaai-data/policies.pdf


In [2]:
!pip -q install "transformers==4.52.4" "accelerate>=0.33" "bitsandbytes>=0.43"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
!pip install fastapi uvicorn pyngrok accelerate -q

In [4]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_id = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(base_id)     # named `tokenizer`, see bug below

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    base_id,
    quantization_config=bnb_cfg,
    torch_dtype=torch.float16,
    device_map={"": 0},          # whole model on GPU 0, GPU 1 stays free for embeddings
    low_cpu_mem_usage=True,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [5]:
def generate_text(prompt, max_new_tokens=300, sample=False):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    gen = dict(do_sample=True, temperature=0.3, top_p=0.95, top_k=50) if sample else dict(do_sample=False)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id, **gen)
    # decode only the NEW tokens (no prompt echo)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [6]:
def generate_chat(question, context, max_new_tokens=300):
    messages = [
        {"role": "system", "content":
            "You answer questions using ONLY the provided context. "
            "Reply in the same language as the question. "
            "If the answer is not in the context, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, repetition_penalty=1.05)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [7]:
!pip install sentence-transformers faiss-cpu langchain langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 6.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.4 MB/s eta 0:00:00a 0:00:01


In [8]:
from sentence_transformers import SentenceTransformer
import faiss

In [9]:
import pandas as pd, numpy as np

DATA_DIR = "/kaggle/input/datasets/abdullahelnoory/bayyaaai-data"   # use the path printed above

df = pd.read_csv(f"{DATA_DIR}/catalog.csv", encoding="utf-8")

df["in_stock"] = df["in_stock"].eq("yes")                  # "yes"/"no" -> True/False
for c in ["tags_en", "tags_ar", "alternative_id"]:
    df[c] = df[c].fillna("")                               # empty cells -> ""
df["tag_list"] = df["tags_en"].apply(lambda s: [t for t in s.split("|") if t])
df = df.set_index("product_id", drop=False)                # df.loc["P003"] works

In [10]:
def product_text(r):
    return (f"{r.name_ar} | {r.name_en} | {r.brand_ar} | {r.category_ar} | {r.category_en} | "
            f"{r.size_ar} | {r.description_ar} | {r.description_en} | {r.tags_ar} | {r.tags_en}")

df["text"] = df.apply(product_text, axis=1)

embedding_model = SentenceTransformer("BAAI/bge-m3", device="cuda:1")
embedding_model.max_seq_length = 512
prod_emb = embedding_model.encode(
    df["text"].tolist(), normalize_embeddings=True, batch_size=16, convert_to_numpy=True
)   # row i of prod_emb belongs to row i of df

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [11]:
import faiss

prod_index = faiss.IndexFlatIP(prod_emb.shape[1])     # inner product = cosine, since embeddings are normalized
prod_index.add(prod_emb.astype("float32"))            # FAISS requires float32
print(prod_index.ntotal)                              # should print 77

77


In [12]:
def build_mask(max_price=None, category=None, must_tags=(), only_in_stock=True):
    mask = pd.Series(True, index=df.index)
    if max_price is not None:
        mask &= df.price_egp <= max_price
    if category:
        mask &= df.category_en.str.contains(category, case=False, regex=False)
    for t in must_tags:
        mask &= df.tag_list.apply(lambda l: t in l)
    if only_in_stock:
        mask &= df.in_stock
    return mask

def search_products(query, k=4, **filters):
    allowed = set(np.where(build_mask(**filters).values)[0])
    if not allowed:
        return df.iloc[[]]

    q = embedding_model.encode([query], normalize_embeddings=True,
                               convert_to_numpy=True).astype("float32")
    scores, ids = prod_index.search(q, len(df))        # rank every product (77 is tiny)

    hits = [(i, s) for i, s in zip(ids[0], scores[0]) if i in allowed][:k]
    out = df.iloc[[i for i, _ in hits]].copy()
    out["score"] = [s for _, s in hits]
    return out

In [13]:
def get_substitute(pid):
    r = df.loc[pid]
    if r.in_stock or not r.alternative_id:
        return None
    return df.loc[r.alternative_id]        # P003 -> P004 (soy milk)

In [14]:
def format_for_llm(rows):
    lines = []
    for r in rows.itertuples():
        stock = "متوفر" if r.in_stock else "غير متوفر"
        unit = f"، {r.price_per_kg_or_l:.0f} جنيه للكيلو/اللتر" if pd.notna(r.price_per_kg_or_l) else ""
        lines.append(f"- [{r.product_id}] {r.name_ar} ({r.brand_ar}، {r.size_ar}): "
                     f"{r.price_egp} جنيه{unit}، {stock}. {r.description_ar} الخصائص: {r.tags_ar}")
    return "\n".join(lines)

In [15]:
faq_chunks = [b.strip() for b in
              open(f"{DATA_DIR}/faq.txt", encoding="utf-8").read().split("\n\n")[1:] if b.strip()]
# -> 30 chunks, each is one "س: ... ج: ..." pair

pol_chunks = []
for sec in open(f"{DATA_DIR}/policies.txt", encoding="utf-8").read().split("## ")[1:]:
    title, *paras = sec.strip().split("\n")
    pol_chunks += [f"{title}: {p}" for p in paras if p.strip()]
# -> 34 chunks, each paragraph keeps its section title for context

import faiss
kb_chunks = faq_chunks + pol_chunks
kb_emb = embedding_model.encode(kb_chunks, normalize_embeddings=True, convert_to_numpy=True)
kb_index = faiss.IndexFlatIP(kb_emb.shape[1])
kb_index.add(kb_emb)

def search_kb(query, k=3):
    q = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    _, ids = kb_index.search(q, k)
    return [kb_chunks[i] for i in ids[0]]

In [16]:
# اختبار قاعدة المنتجات
for q in ["عايز لبن بدون لاكتوز", "أرز للمحشي", "حاجة نباتية للفطار", "something for tea"]:
    print("🔎", q)
    print(search_products(q, k=3, only_in_stock=False)[["name_ar", "price_egp", "in_stock", "score"]], "\n")

# اختبار قاعدة السياسات والأسئلة الشائعة
for q in ["الحد الأدنى للطلب كام؟", "إزاي أرجّع منتج؟", "بتوصلوا إسكندرية؟", "لو الطلب اتأخر يحصل إيه؟"]:
    print("🔎", q)
    for c in search_kb(q, 3):
        print("   -", c[:100])

🔎 عايز لبن بدون لاكتوز
                         name_ar  price_egp  in_stock     score
product_id                                                     
P003        لبن خالٍ من اللاكتوز         78     False  0.620409
P004           لبن صويا بدون سكر         85      True  0.573600
P002              لبن قليل الدسم         46      True  0.552064 

🔎 أرز للمحشي
              name_ar  price_egp  in_stock     score
product_id                                          
P012        أرز بسمتي        110      True  0.591990
P015            شعرية         16      True  0.583350
P011         أرز مصري         42      True  0.571685 

🔎 حاجة نباتية للفطار
                 name_ar  price_egp  in_stock     score
product_id                                             
P049          كورن فليكس         95      True  0.640679
P073        مكسرات مشكلة        140      True  0.547073
P025           سمن نباتي         95      True  0.538618 

🔎 something for tea
                        name_ar  price_egp  in_stock

In [17]:
import json, re, torch
from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.output_parsers import PydanticOutputParser
from typing import Any, Literal, Optional

GEN = {"max_new_tokens": 300, "sample": False}   # settings read by the LLM on every call

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None, run_manager: Any = None, **kwargs) -> str:
        return generate_text(prompt, max_new_tokens=GEN["max_new_tokens"], sample=GEN["sample"])

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [18]:
from pydantic import BaseModel, Field

INTENTS = ["product_search", "price_inquiry", "availability_check", "comparison",
           "policy_or_faq", "order_issue", "ready_to_buy", "other"]

NEXT_ACTIONS = ["send_reply", "ask_clarifying_question", "offer_substitute",
                "confirm_order", "escalate_to_human", "follow_up_later"]

class CustomerIntent(BaseModel):
    intent: Literal[tuple(INTENTS)] = Field(description="The main intent of the customer")
    language: str = Field(description="e.g. 'egyptian_arabic', 'msa', 'english'")
    sentiment: Literal["positive", "neutral", "negative"]
    needs_product: bool = Field(description="True only if a product recommendation would help")
    search_query: str = Field(description="Short product search phrase, '' if no product is needed")
    max_price: Optional[float] = Field(default=None, description="Budget in EGP if stated, else null")
    category: Optional[str] = Field(default=None, description="One of the allowed categories, or null")
    must_tags: list[str] = Field(default_factory=list, description="Required tags from the allowed list, else []")

class ProductChoice(BaseModel):
    product_id: Optional[str] = Field(default=None, description="ID of the chosen candidate, e.g. 'P004', or null")

class ReasonOut(BaseModel):
    reason: str = Field(description="1-2 sentences on why this product fits, or why none was chosen")

class ReplyOut(BaseModel):
    suggested_reply: str = Field(description="The message the sales rep can send to the customer")
    next_action: Literal[tuple(NEXT_ACTIONS)]

In [20]:
from langchain_core.output_parsers import PydanticOutputParser

intent_parser    = PydanticOutputParser(pydantic_object=CustomerIntent)
recommend_parser = PydanticOutputParser(pydantic_object=ProductChoice)
reason_parser    = PydanticOutputParser(pydantic_object=ReasonOut)
reply_parser     = PydanticOutputParser(pydantic_object=ReplyOut)

def extract_json_block(text):
    blocks = re.findall(r'```json\s*(.*?)\s*```', text, re.DOTALL)
    if blocks:
        return f"```json\n{blocks[-1]}\n```"
    s, e = text.find("{"), text.rfind("}")          # fallback: model forgot the fences
    return text[s:e + 1] if s != -1 and e > s else text

def run_parsed(chain, parser, max_new_tokens=300, **inputs):
    """chain.run -> extract JSON -> Pydantic parse. Retry with light sampling if parsing fails."""
    err = None
    GEN["max_new_tokens"] = max_new_tokens
    try:
        for attempt in range(3):
            GEN["sample"] = attempt > 0
            try:
                return parser.parse(extract_json_block(chain.run(inputs)))
            except Exception as e:
                err = e
    finally:
        GEN["sample"] = False
    raise err

In [21]:
ALL_CATEGORIES = sorted(df.category_en.dropna().unique().tolist())
ALL_TAGS = sorted({t for tags in df.tag_list for t in tags})

def retrieve_products(message, intent):
    if not intent.needs_product:
        return df.iloc[[]]

    category = intent.category if intent.category in ALL_CATEGORIES else None
    tags = [t for t in intent.must_tags if t in ALL_TAGS]
    query = intent.search_query or message

    hits = df.iloc[[]]
    for f in [dict(max_price=intent.max_price, category=category, must_tags=tags),
              dict(max_price=intent.max_price, category=category),
              dict(max_price=intent.max_price),
              dict()]:                                   # strict -> relaxed
        hits = search_products(query, k=4, only_in_stock=False, **f)
        if len(hits):
            break

    # add substitutes for out-of-stock hits
    extra = [get_substitute(pid) for pid in hits.index if not hits.loc[pid, "in_stock"]]
    extra = [r for r in extra if r is not None and r.product_id not in hits.index]
    if extra:
        hits = pd.concat([hits, pd.DataFrame(extra).assign(score=np.nan)])
    return hits

def retrieve_policies(message):
    return "\n".join(f"- {c}" for c in search_kb(message, k=3))

In [23]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# Chain 1: Detect customer intent
intent_prompt = PromptTemplate(
    input_variables=["message"],
    partial_variables={
        "categories": json.dumps(ALL_CATEGORIES, ensure_ascii=False),
        "tags": json.dumps(ALL_TAGS, ensure_ascii=False),
        "format_instructions": intent_parser.get_format_instructions(),
    },
    template="""You are the intent-analysis step of an AI assistant for a grocery sales team in Egypt.
Customers write in Egyptian Arabic, Modern Standard Arabic or English.

Intent labels:
- product_search: looking for something to buy
- price_inquiry: asking the price of a product
- availability_check: asking if a product is in stock
- comparison: comparing products / asking for the best one
- policy_or_faq: delivery, returns, payment, minimum order, hours...
- order_issue: complaint about an existing order (late, wrong, damaged)
- ready_to_buy: wants to place an order now
- other: greeting, thanks, unrelated

Rules:
- needs_product is true only if recommending a product would help.
- search_query: short phrase in the customer's language; "" if needs_product is false.
- max_price: number in EGP only if a budget is stated, else null.
- category: exactly one of {categories}, or null.
- must_tags: only tags from {tags}, only if clearly required (e.g. lactose-free), else [].

Respond ONLY in JSON format as follows:
{format_instructions}

Customer message:
"{message}"
"""
)
intent_chain = LLMChain(llm=llm, prompt=intent_prompt)


# Chain 2: Recommend the best product
recommend_prompt = PromptTemplate(
    input_variables=["message", "intent_json", "catalog_ctx"],
    partial_variables={"format_instructions": recommend_parser.get_format_instructions()},
    template="""You are the product-selection step of an AI sales assistant.
Choose the ONE best product for the customer from the candidate list.

Rules:
- Use only IDs that appear in the candidate list. Never invent an ID.
- Only choose products that are in stock (متوفر). Never choose one marked غير متوفر.
- Respect the customer's budget and requirements.
- If nothing fits, return null.

Respond ONLY in JSON format as follows:
{format_instructions}

Customer message:
"{message}"

Detected intent:
{intent_json}

Candidate products:
{catalog_ctx}
"""
)
recommend_chain = LLMChain(llm=llm, prompt=recommend_prompt)


# Chain 3: Explain the reason
reason_prompt = PromptTemplate(
    input_variables=["message", "intent_json", "product_ctx", "kb_ctx"],
    partial_variables={"format_instructions": reason_parser.get_format_instructions()},
    template="""You are the reasoning step of an AI sales assistant.
In 1-2 short sentences, explain to the sales rep why the selected product fits this customer
(budget, features, size, stock, relevant policy). If no product was selected, explain why
(policy question, nothing matches, out of stock...).

Rules:
- Use ONLY facts from the catalog and policy context below. Do not invent anything.
- Write in the same language as the customer's message.

Respond ONLY in JSON format as follows:
{format_instructions}

Customer message:
"{message}"

Detected intent:
{intent_json}

Selected product:
{product_ctx}

Relevant policies / FAQ:
{kb_ctx}
"""
)
reason_chain = LLMChain(llm=llm, prompt=reason_prompt)


# Chain 4: Write the reply + next action
reply_prompt = PromptTemplate(
    input_variables=["message", "intent_json", "product_ctx", "kb_ctx", "reason"],
    partial_variables={"format_instructions": reply_parser.get_format_instructions()},
    template="""You are the reply-writing step of an AI sales assistant. Write the message a sales rep can send
to the customer, and choose the next action for the sales team.

Reply rules:
- Same language and dialect as the customer (Egyptian Arabic -> simple Egyptian Arabic).
- Warm, professional, at most 4 short sentences.
- Use ONLY facts from the product and policy context. Never invent prices, discounts, delivery times or policies.
- If a product is selected, mention its name, price and availability, and end with a clear call to action.
- Policy question: answer from the policy context; if it's not there, say the team will confirm.
- Upset customer / order problem: apologise briefly and offer help.

next_action options:
- send_reply: the reply fully answers the customer
- ask_clarifying_question: request too vague to recommend anything
- offer_substitute: requested item unavailable, alternative offered
- confirm_order: customer is ready to buy
- escalate_to_human: complaint, sensitive issue, or missing information
- follow_up_later: customer is undecided

Respond ONLY in JSON format as follows:
{format_instructions}

Customer message:
"{message}"

Detected intent:
{intent_json}

Selected product:
{product_ctx}

Relevant policies / FAQ:
{kb_ctx}

Reason for the choice:
{reason}
"""
)
reply_chain = LLMChain(llm=llm, prompt=reply_prompt)

/tmp/ipykernel_47/2829802034.py:39: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  intent_chain = LLMChain(llm=llm, prompt=intent_prompt)


In [24]:
def run_all(message: str):
    print(f"Customer: {message}")

    # Chain 1: intent
    intent = run_parsed(intent_chain, intent_parser, max_new_tokens=250, message=message)
    intent_json = intent.model_dump_json()
    print(f"\nIntent:\n{intent_json}")

    # RAG
    candidates = retrieve_products(message, intent)
    kb_ctx = retrieve_policies(message)
    catalog_ctx = format_for_llm(candidates) if len(candidates) else "No candidate products."

    # Chain 2: recommend (+ guardrail against invented / out-of-stock IDs)
    pid = None
    if len(candidates):
        pid = run_parsed(recommend_chain, recommend_parser, message=message,
                         intent_json=intent_json, max_new_tokens=80, catalog_ctx=catalog_ctx).product_id
        if pid is not None and not (pid in candidates.index and candidates.loc[pid, "in_stock"]):
            sub = get_substitute(pid) if pid in df.index else None
            if sub is not None and sub.in_stock:
                pid = sub.product_id
            else:
                in_stock = candidates[candidates.in_stock]
                pid = in_stock.index[0] if len(in_stock) else None

    product_ctx = format_for_llm(df.loc[[pid]]) if pid else "No product selected."
    print(f"\nRecommended product ID: {pid}")

    # Chain 3: reason
    reason = run_parsed(reason_chain, reason_parser, message=message, max_new_tokens=200, intent_json=intent_json,
                        product_ctx=product_ctx, kb_ctx=kb_ctx).reason
    print(f"\nReason:\n{reason}")

    # Chain 4: reply + next action
    reply = run_parsed(reply_chain, reply_parser, message=message, max_new_tokens=350, intent_json=intent_json,
                       product_ctx=product_ctx, kb_ctx=kb_ctx, reason=reason)

    product = None
    if pid:
        r = df.loc[pid]
        product = {"product_id": pid, "name": r.name_ar,
                   "price_egp": float(r.price_egp), "in_stock": bool(r.in_stock)}

    result = {
        "customer_intent": intent.intent,
        "recommended_product": product,
        "reason": reason,
        "suggested_reply": reply.suggested_reply,
        "next_action": reply.next_action,
    }
    print("\nFinal output:")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

In [25]:
run_all("عايز لبن بدون لاكتوز")       # P003 is out of stock -> should offer the soy milk substitute
run_all("الحد الأدنى للطلب كام؟")      # policy question, no product
run_all("الطلب اتأخر بقاله يومين وده مش مقبول!")   # complaint -> escalate

Customer: عايز لبن بدون لاكتوز


/tmp/ipykernel_47/2328290661.py:23: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  return parser.parse(extract_json_block(chain.run(inputs)))



Intent:
{"intent":"product_search","language":"egyptian_arabic","sentiment":"positive","needs_product":true,"search_query":"لبن بدون لاكتوز","max_price":null,"category":null,"must_tags":["lactose_free"]}

Recommended product ID: P004

Reason:
هذا المنتج مناسب لك لأنك طلبت 'لبن بدون لاكتوز'، وهو 'لبن صويا خالٍ من اللاكتوز'، بالإضافة إلى أنه 'بدون سكر'، مما يجعله خاليًا من المكونات التي قد تثير حساسية أو تقييد.

Final output:
{
  "customer_intent": "product_search",
  "recommended_product": {
    "product_id": "P004",
    "name": "لبن صويا بدون سكر",
    "price_egp": 85.0,
    "in_stock": true
  },
  "reason": "هذا المنتج مناسب لك لأنك طلبت 'لبن بدون لاكتوز'، وهو 'لبن صويا خالٍ من اللاكتوز'، بالإضافة إلى أنه 'بدون سكر'، مما يجعله خاليًا من المكونات التي قد تثير حساسية أو تقييد.",
  "suggested_reply": "يوجد لدينا 'لبن صويا خالٍ من اللاكتوز' (نيلي، 1 لتر) 85 جنيه. مشروب صويا نباتي بدون سكر مضاف، بديل للبن الحيواني. يلبى طلبك 'لبن بدون لاكتوز'. هل تريد طلبه؟",
  "next_action": "confirm_ord

{'customer_intent': 'order_issue',
 'recommended_product': None,
 'reason': "No product selected as the customer's issue is about delayed order, not a specific product.",
 'suggested_reply': 'نأسف على التأخير. إذا تأخر الطلب عن 60 دقيقة، يمكنك استرداد رسوم التوصيل أو كوبون 30 جنيها. كيف نرغب في المساعدة؟',
 'next_action': 'ask_clarifying_question'}

In [26]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
API_KEY = user_secrets.get_secret("API_KEY")

In [27]:
import io, contextlib, threading
from fastapi import FastAPI, Form, Header, HTTPException

app = FastAPI()
llm_lock = threading.Lock()   # one request at a time: one GPU model + the shared GEN settings

def check_key(authorization):
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/answer")
def answer(
    prompt: str = Form(...),
    authorization: str = Header(None),
):
    check_key(authorization)
    if not prompt.strip():
        raise HTTPException(422, "Prompt is empty")

    try:
        with llm_lock, contextlib.redirect_stdout(io.StringIO()):   # silence run_all's prints
            result = run_all(prompt)
    except Exception as e:
        raise HTTPException(502, f"Model returned invalid output, please try again. ({type(e).__name__})")

    return {
        "response": result["suggested_reply"],   # for a simple chat UI that only reads "response"
        "mode": "sales_assistant",
        **result,                                # customer_intent, recommended_product, reason, suggested_reply, next_action
    }

In [28]:
import uvicorn, socket, time
from pyngrok import ngrok, conf

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

ngrok.kill()                      # close any old tunnel (the free plan allows only one)
port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://b73e-136-111-57-131.ngrok-free.app                                         


INFO:     Started server process [47]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:56883 (Press CTRL+C to quit)


INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
INFO:     156.201.67.80:0 - "POST /answer HTTP/1.1" 200 OK
